# 12 · Lectura de placas con una CNN de TensorFlow/Keras (Deep Learning, requisito E.4)

Se ejecuta en **Google Colab con GPU** (Entorno de ejecución → Cambiar tipo → T4). Entrena **desde cero** una red convolucional en **TensorFlow/Keras** que lee los 7 caracteres de una placa boliviana
de formato A4 (4 dígitos + 3 letras) y la exporta a **TFLite** (móvil y navegador con `@tensorflow/tfjs-tflite`), para la IA local de la app de portería.

**Diseño.** Como A4 fija el formato (4 dígitos y 3 letras), la red tiene **una salida de 7 × 36** (softmax por posición sobre dígitos+letras); al decodificar se restringe a dígitos en las 4 primeras posiciones y a letras en las 3 últimas. No hace falta CTC y la salida única evita que TFLite reordene tensores.
La confianza de la placa es el producto de las probabilidades máximas (se usa con el umbral 0,85 de A4).

**Datos.** Placas sintéticas (`ml/src/placas_sinteticas.py`, texto conocido) con aumentos fuertes: giro, desenfoque, luz, ruido, baja resolución, recorte e inversión de contraste. **Son optimistas**:
la prueba honesta es sobre los **10 recortes reales** del notebook 10 (se suben como `recortes.zip`, no se versionan, I6). Se compara con EasyOCR + formato (6/10, notebook 10).

**Archivos a subir:** `ml.zip` (carpeta `ml/` del repo) y `recortes.zip` (los 10 recortes + `verdad.json`), los mismos del notebook 11.

In [ ]:
import os, zipfile, json
from google.colab import files

subidos = files.upload()     # ml.zip + recortes.zip
os.makedirs("/content/condohub-ia", exist_ok=True)
os.makedirs("/content/recortes", exist_ok=True)
for nombre in subidos:
    zipfile.ZipFile(nombre).extractall("/content/condohub-ia" if nombre.lower().startswith("ml") else "/content/recortes")
os.chdir("/content/condohub-ia")
verdad = json.load(open("/content/recortes/verdad.json", encoding="utf-8"))
print(len(verdad), "recortes reales")

In [ ]:
import random, string, time, numpy as np, tensorflow as tf
from tensorflow import keras
from PIL import Image
from ml.src import placas_sinteticas as ps, placas
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)

## 1 · Datos: placas sintéticas con aumentos

La red lee la **franja inferior** de la placa (donde están los caracteres; el rótulo «BOLIVIA» queda fuera), en gris, a **32 × 128**.
Alfabeto de 36 símbolos (0–9, A–Z); la etiqueta de cada posición es su índice.

In [ ]:
ALTO, ANCHO = 32, 128
ALFA = string.digits + string.ascii_uppercase      # 0-9 = 0..9, A-Z = 10..35

def etiqueta(texto):
    return [ALFA.index(c) for c in texto]

def decodificar(prob):
    """prob: [7, 36]. Dígitos en las posiciones 0-3 y letras en 4-6; confianza = producto de máximos."""
    p = prob.copy(); p[:4, 10:] = 0; p[4:, :10] = 0
    p = p / p.sum(axis=1, keepdims=True)
    return "".join(ALFA[k] for k in p.argmax(1)), float(p.max(axis=1).prod())

def preparar(im, desde=0.4):
    """Franja inferior en gris, 32x128, valores 0..1."""
    im = im.convert("L")
    im = im.crop((0, int(desde * im.height), im.width, im.height))
    return np.asarray(im.resize((ANCHO, ALTO), Image.BICUBIC), dtype=np.float32) / 255.0

def generar(n, rng, fuerte=True):
    X = np.zeros((n, ALTO, ANCHO, 1), np.float32)
    Y = np.zeros((n, 7), np.int32)
    for i in range(n):
        t = ps.texto_aleatorio(rng)
        im = ps.dibujar(t, rng, degradar=True)
        if fuerte:                                    # aumentos extra: placas pequeñas, sucias, mal encuadradas
            w = rng.randint(90, 180)
            im = im.resize((w, w // 2), Image.BILINEAR).resize((360, 180), Image.BILINEAR)
            im = im.crop((rng.randint(0, 24), rng.randint(0, 12), 360 - rng.randint(0, 24), 180 - rng.randint(0, 12)))
        a = preparar(im, desde=rng.uniform(0.3, 0.5) if fuerte else 0.4)
        if fuerte:
            a = np.clip(a + np.random.normal(0, rng.uniform(0, 0.06), a.shape), 0, 1)
            if rng.random() < 0.25:
                a = 1 - a                              # contraste invertido (placas oscuras)
        X[i, :, :, 0] = a
        Y[i] = etiqueta(t)
    return X, Y

Xtr, Ytr = generar(40000, random.Random(SEMILLA))
Xva, Yva = generar(4000, random.Random(SEMILLA + 1))
Xte, Yte = generar(300, random.Random(SEMILLA + 2), fuerte=False)     # 300 placas como las del notebook 04 (sin aumentos extra)
print(Xtr.shape, Xva.shape, Xte.shape)

## 2 · Red convolucional (Keras)

Tres bloques Conv-BN-ReLU-MaxPool, una capa densa y una salida 7×36 con softmax por posición. Pocos parámetros a propósito: tiene que correr en el celular.

In [ ]:
def construir():
    x = entrada = keras.Input((ALTO, ANCHO, 1))
    for f in (32, 64, 128):
        x = keras.layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.ReLU()(x)
        x = keras.layers.MaxPooling2D(2)(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(7 * 36)(x)
    x = keras.layers.Reshape((7, 36))(x)
    return keras.Model(entrada, keras.layers.Softmax(axis=-1, name="probabilidades")(x), name="placas_cnn")

modelo = construir()
modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.summary()

## 3 · Entrenamiento

In [ ]:
cb = [keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor="val_loss"),
      keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5)]
hist = modelo.fit(Xtr, Ytr, validation_data=(Xva, Yva), epochs=120, batch_size=128, callbacks=cb, verbose=2)
print("épocas:", len(hist.history["loss"]))

## 4 · Evaluación

Tres conjuntos: validación sintética con aumentos, **300 placas sintéticas** (comparables con el 99,67 % de EasyOCR del notebook 04) y las **10 placas reales** (comparables con 6/10 del notebook 10).
En las reales importa sobre todo cuántas lecturas **equivocadas** se aceptarían con el umbral de A4.

In [ ]:
def leer(X):
    lect = [decodificar(p) for p in modelo.predict(X, verbose=0)]
    return [t for t, _ in lect], np.array([c for _, c in lect])

def exactitud(X, Y):
    t, _ = leer(X)
    return placas.metricas_lectura(["".join(ALFA[k] for k in f) for f in Y], t)

m_va = exactitud(Xva, Yva)
m_te = exactitud(Xte, Yte)
print("validación sintética (con aumentos):", m_va)
print("300 sintéticas (sin aumentos):", m_te)

In [ ]:
reales = [v["real"] for v in verdad]
Xr = np.stack([preparar(Image.open(f"/content/recortes/{v['archivo']}"), desde=0.4)[..., None] for v in verdad])
textos_r, conf_r = leer(Xr)
UMBRAL = 0.85
for r, t, c in zip(reales, textos_r, conf_r):
    print(f"real {r:8s} leída {t}  conf {c:.2f}  {'OK' if t == r else 'x'}")
exactas = sum(t == r for t, r in zip(textos_r, reales))
a4 = [i for i, r in enumerate(reales) if placas.es_placa_valida(r)]      # la placa antigua (3 dígitos) no cumple A4
print(f"\nexactas {exactas}/{len(reales)} | sobre las {len(a4)} con formato A4: {sum(textos_r[i] == reales[i] for i in a4)}/{len(a4)}")
print("con confianza >=", UMBRAL, ":", sum(c >= UMBRAL for c in conf_r),
      "| equivocadas aceptadas:", sum((c >= UMBRAL) and t != r for t, r, c in zip(textos_r, reales, conf_r)))

## 5 · Lectura del resultado (honesto)

El modelo es **solo sintético**. Con 10 placas reales no se puede entrenar y medir a la vez: la cifra real de arriba es la que se declara. Si es floja, la mejora válida es reentrenar con más
recortes reales (fotos propias de placas) y evaluar con otros distintos. El notebook 11 (EasyOCR → ONNX) sigue siendo la referencia de comparación.

## 6 · Exportar a TFLite (móvil y navegador)

El `.tflite` corre en el celular con TFLite y en el navegador con `@tensorflow/tfjs-tflite`. No se convierte a TF.js porque el paquete `tensorflowjs` no es compatible con el Colab actual (NumPy 2, Python 3.13).

In [ ]:
os.makedirs("/content/salida", exist_ok=True)
modelo.save("/content/salida/placas_cnn.keras")

conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
tfl_fp32 = conv.convert()
open("/content/salida/placas_cnn_fp32.tflite", "wb").write(tfl_fp32)
conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
conv.optimizations = [tf.lite.Optimize.DEFAULT]                 # cuantización dinámica
tfl_q = conv.convert()
open("/content/salida/placas_cnn_int8.tflite", "wb").write(tfl_q)

# TF.js no se convierte: tensorflowjs choca con NumPy 2 / Python 3.13 / packaging en Colab y degrada el entorno.
# La PWA ejecuta el .tflite en el navegador con @tensorflow/tfjs-tflite, sin conversión.
for f in sorted(os.listdir("/content/salida")):
    p = os.path.join("/content/salida", f)
    print(f, round(os.path.getsize(p) / 1e6, 2) if os.path.isfile(p) else "(carpeta)", "MB")

## 7 · Paridad Keras ↔ TFLite y velocidad

In [ ]:
def leer_tflite(modelo_bytes, X):
    it = tf.lite.Interpreter(model_content=modelo_bytes); it.allocate_tensors()
    ent, sal = it.get_input_details()[0], it.get_output_details()[0]
    res = []
    for x in X:
        it.set_tensor(ent["index"], x[None]); it.invoke()
        res.append(decodificar(it.get_tensor(sal["index"])[0])[0])
    return res

for nombre, b in (("fp32", tfl_fp32), ("int8", tfl_q)):
    lect = leer_tflite(b, Xr)
    print(f"texto igual Keras/TFLite {nombre}:", sum(a == k for a, k in zip(lect, textos_r)), "/", len(Xr))
t0 = time.time(); leer_tflite(tfl_q, Xte[:100]); print("TFLite int8:", round((time.time() - t0) / 100 * 1000, 1), "ms por placa (CPU de Colab)")

## 8 · Guardar métricas y descargar

Las métricas (sin placas ni lecturas, I6) van a `ml/experiments/2026-10-18_placas_tensorflow/metrics.json`. Los modelos (`.keras`, `.tflite`) **no van a git**: al Drive del equipo.

In [ ]:
metricas = {
    "modelo": "CNN Keras: 3 bloques conv + 7 cabezas softmax (4x10, 3x26)", "tensorflow": tf.__version__, "semilla": SEMILLA,
    "parametros": int(modelo.count_params()), "entrenamiento": "40000 placas sinteticas con aumentos",
    "sintetico_validacion": m_va, "sintetico_300": m_te,
    "reales_10": {"exactas": int(exactas), "total": len(reales), "umbral": UMBRAL,
                  "aceptadas": int(sum(c >= UMBRAL for c in conf_r)),
                  "equivocadas_aceptadas": int(sum((c >= UMBRAL) and t != r for t, r, c in zip(textos_r, reales, conf_r)))},
    "tflite_int8_mb": round(len(tfl_q) / 1e6, 2),
}
json.dump(metricas, open("/content/salida/metrics.json", "w"), indent=2, ensure_ascii=False)
!cd /content && zip -qr salida.zip salida
from google.colab import files
files.download("/content/salida.zip")
print(json.dumps(metricas, indent=2, ensure_ascii=False))